# Figure S1 — What the ENCODE BPNet models rely on: control track, motifs and sequence composition

Test-set counts AUROC/AUPRC (ENCODE's test folds, peaks vs GC-matched nonpeaks, held-out fold model per region) under
sequence perturbations, with the region's own control track (`ctl`) and with an all-zero control (`noctl`):
- **p0ko / p0rand** — the TF-MoDISco pattern_0 Fi-NeMo sites of each test peak shuffled / replaced by random bases;
  **othko** — the other positive patterns shuffled, pattern_0 kept
- **shufpk** — dinucleotide-shuffled peaks vs real nonpeaks; **shufall** — both shuffled
- **mix** — every region scored on another region's sequence, with its own control (identity mix)

Inputs: `data/bpnet_big_table.parquet` (one row per model, `pipeline/01_auroc/build_big_table.py`),
`data/examples/HepG2_{SP4,CTCF}/` (per-region test-set predictions), the ENCODE TF-MoDISco reports of HepG2 SP4 and
CTCF under `$BPNET_DATA/PAPER/HepG2/`, `metadata/hocomoco_v14_cluster_list.tsv` (TF families). No GPU needed.

In [ ]:
import sys
sys.path.insert(0, '../modules')        # run from notebooks/
import paths                            # data root, genome, repo locations (env BPNET_DATA, MISHA_HG38)
from plot_setup import fig_path         # editable PDF text; figures/ in the repository
%config InlineBackend.figure_format = 'retina'

In [ ]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import circlify
from matplotlib import cm, colors
from perf import ALL_CONDS, plot_ko_curves, plot_modisco_report, plot_motifko_hexbin, tf_subfamilies

mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Helvetica', 'Arial', 'Nimbus Sans', 'DejaVu Sans'],
    'axes.unicode_minus': False,      # plain hyphen; avoids a missing-glyph box for U+2212
    'savefig.bbox': 'tight',
    'savefig.transparent': True,
})

## Two example models: HepG2 SP4 and HepG2 CTCF
Fold-mean test-set ROC and PR curves, all test peaks.

In [ ]:
MOTIF_KO = ['WT', 'p0ko', 'p0rand', 'othko']
for tf in ['SP4', 'CTCF']:
    fig, mean = plot_ko_curves(f'HepG2/{tf}', 'paper', conds=MOTIF_KO, peaks='all')
    fig.savefig(fig_path(f'hepg2_{tf.lower()}_motif_curves.pdf'))

In [ ]:
for tf in ['CTCF', 'SP4']:
    fig, tab = plot_modisco_report(f'HepG2/{tf}', 'paper')     # first 5 TF-MoDISco counts-head patterns
    fig.savefig(fig_path(f'{tf}_logos.pdf'))

In [ ]:
for tf in ['CTCF', 'SP4']:
    fig, ax = plot_motifko_hexbin(f'HepG2/{tf}', 'paper')     # test peaks: own control vs no control / shuffled / mixed
    fig.savefig(fig_path(f'{tf}_scatters.pdf'))

In [ ]:
SEQ_PERTURB = ['WT', 'shufpk', 'shufall', 'mix']
for tf in ['SP4', 'CTCF']:
    fig, mean = plot_ko_curves(f'HepG2/{tf}', 'paper', conds=SEQ_PERTURB, peaks='all')
    fig.savefig(fig_path(f'hepg2_{tf.lower()}_peak_perturb_curves.pdf'))

## All models
`big`: one row per model (2,282). TF families: HOCOMOCO v14 motif-cluster subfamily of the TF
(`Not In Cluster` -> NaN), plus a few hand-made name groups (`misc_cat`).

In [ ]:
big = pd.read_parquet(paths.DATA / 'bpnet_big_table.parquet')
out = big.merge(tf_subfamilies(), on='tf_name', how='left', validate='many_to_one')

In [ ]:
# models whose prediction on pattern_0-containing peaks collapses when pattern_0 is replaced by random bases
cols = ['tf_name', 'cell_line', 'n_peaks', 'auroc_p0rand_ctl_allpk', 'auroc_noctl', 'auroc_mix', 'r_ctl__noctl',
        'r_ctl__mix_ctl', 'r_ctl__p0rand_ctl', 'bpnet_model_qc_status', 'auroc_ctl', 'encode_pearsonr_peaks']
motif_dependent = (out[(out.auroc_p0rand_ctl_p0pk <= 0.8) & (out.bpnet_model_qc_status != 'archived')]
                   .loc[:, cols].sort_values(by=['tf_name', 'auroc_p0rand_ctl_allpk']).round(2))
motif_dependent.to_csv(fig_path('motif_dependent.csv'), index=False)
len(motif_dependent)

In [ ]:
# TFs whose models drop to AUROC <= 0.75 when pattern_0 is randomised (all test peaks), circle area = number of models
cnt = (out[(out.auroc_p0rand_ctl_allpk <= 0.75) & (out.bpnet_model_qc_status != 'archived')].tf_name
       .fillna('').str.strip().replace('', '(no name)').value_counts())
print(cnt.sum(), 'models,', len(cnt), 'TFs')

circles = circlify.circlify([{'id': k, 'datum': int(v)} for k, v in cnt.items()],
                            show_enclosure=False, target_enclosure=circlify.Circle(x=0, y=0, r=1))
norm = colors.LogNorm(vmin=cnt.min(), vmax=cnt.max())
cmap = plt.get_cmap('cividis')
min_label = 2                                   # leave singletons unlabelled

fig, ax = plt.subplots(figsize=(7, 7), dpi=200)
ax.axis('off')
lim = max(max(abs(c.x) + c.r, abs(c.y) + c.r) for c in circles)
ax.set(xlim=(-lim, lim), ylim=(-lim, lim), aspect='equal')
for c in circles:
    name, n = c.ex['id'], c.ex['datum']
    ax.add_patch(plt.Circle((c.x, c.y), c.r, facecolor=cmap(norm(n)), edgecolor='white', linewidth=0.8))
    if n >= min_label:
        ax.text(c.x, c.y, f'{name}\n{n}', ha='center', va='center', fontsize=np.clip(c.r * 60, 5, 12),
                color='white' if norm(n) < 0.6 else 'black')
fig.colorbar(cm.ScalarMappable(norm=norm, cmap=cmap), ax=ax, shrink=0.5, label='count')
fig.savefig(fig_path('subfam_bubbles.pdf'), bbox_inches='tight')

In [ ]:
pivot_big = (big.loc[:, ['auroc_ctl', 'auroc_shufpk_ctl', 'auroc_p0rand_ctl_allpk', 'bpnet_model_qc_status']]
             .set_index('bpnet_model_qc_status').stack().reset_index())
g = sns.displot(data=pivot_big, x=0, hue='level_1', col='bpnet_model_qc_status', kind='hist', height=10, aspect=1,
                stat='percent', common_norm=False, cumulative=False, multiple='stack', alpha=.9, bins=60,
                element='step', palette='mako')
g.set(xlim=(0.45, 1.1))
g.savefig(fig_path('auroc_hist.pdf'))

## By TF family
Share of the above-chance AUROC that survives a perturbation: (AUROC_perturbed − 0.5) / (AUROC_ctl − 0.5).
`ctl_share` = identity-mixed sequences, own control. Grey: all models; coloured: the family's non-archived models;
yellow: HepG2 CTCF / SP4.

In [ ]:
out['dinuc_ctrl_share'] = (out.auroc_shufpk_ctl_allpk - 0.5) / (out.auroc_ctl - 0.5)
out['noctrl_share'] = (out.auroc_noctl - 0.5) / (out.auroc_ctl - 0.5)

out['misc_cat'] = None
out.loc[out.tf_name.str.fullmatch(r'KLF\d+|SP\d+|MAZ'), 'misc_cat'] = 'KLF|SP|MAZ'    # KLF1-17, SP1-9, MAZ
out.loc[out.tf_name.apply(lambda x: 'FOX' in x), 'misc_cat'] = 'FOX_'
out.loc[out.tf_name.apply(lambda x: 'SOX' in x), 'misc_cat'] = 'SOX_'
out.loc[out.tf_name.apply(lambda x: 'HOX' in x), 'misc_cat'] = 'HOX_'
out.loc[out.tf_name.apply(lambda x: 'CTCF' in x), 'misc_cat'] = 'CTCF_'

families = list(out.misc_cat.dropna().unique()) + ['ATF2', 'HAIRY', 'ETS-like', 'Fos', 'RAR (NR1B)', 'CEBP', 'E2F',
                                                   'GATA double', 'YY1-like', 'Large MAF']
nctcf = out[(out.set == 'PAPER') & (out.cell_line == 'HepG2') & (out.tf_name == 'CTCF')]
nsp4 = out[(out.set == 'PAPER') & (out.cell_line == 'HepG2') & (out.tf_name == 'SP4')]


def family_panels(x, y, c, cmap, vmin, vmax, xlabel, ylabel, clabel, lim=None, guides=True):
    fig, axes = plt.subplots(3, 5, sharex=True, sharey=True, figsize=(20, 12), constrained_layout=True, dpi=200)
    for cfam, ax in zip(families, axes.flatten()):
        subset = out[((out.misc_cat == cfam) | (out.subfam == cfam)) & (out.bpnet_model_qc_status != 'archived')]
        if cfam == 'CTCF_':
            ax.scatter(nctcf[x], nctcf[y], s=100, c='yellow', zorder=20)
        if cfam == 'KLF|SP|MAZ':
            ax.scatter(nsp4[x], nsp4[y], s=100, c='yellow', zorder=20)
        ax.scatter(out[x], out[y], color='gainsboro', alpha=.6, s=40, rasterized=True)
        s = ax.scatter(subset[x], subset[y], c=subset[c], vmin=vmin, vmax=vmax, alpha=.6, s=40, ec='k', cmap=cmap,
                       rasterized=True)
        if guides:
            ax.axvline(.5, c='k', ls=':')
            ax.axhline(.5, c='k', ls=':')
        if lim:
            ax.set_xlim(lim)
            ax.set_ylim(lim)
        ax.set(title=cfam, xlabel=xlabel, ylabel=ylabel)
    fig.colorbar(s, label=clabel)
    return fig

In [ ]:
fig = family_panels('dinuc_ctrl_share', 'ctl_share', 'noctrl_share', 'Reds', 0, 1,
                    '(AUROC DinucShuff - 0.5)/(AUROC - 0.5)', '(AUROC MixedIdentity - 0.5)/(AUROC - 0.5)',
                    '(AUROC No Control - 0.5)/(AUROC - 0.5)', lim=[0, 1.1])
fig.savefig(fig_path('auroc.pdf'))

In [ ]:
fig = family_panels('auprc_shufpk_ctl', 'auprc_mix', 'auprc_noctl', 'Greens', 0.2, 1,
                    'AUPRC DinucShuff', 'AUPRC MixIdentity', 'AUPRC No Control', lim=[0.17, 1.1])
fig.savefig(fig_path('auprc.pdf'))

In [ ]:
# Pearson r over test peaks between the no-control prediction and the same peaks with pattern_0 randomised (x) or
# fully dinucleotide shuffled (y)
fig = family_panels('r_noctl__p0rand_noctl', 'r_noctl__shuf_noctl', 'auroc_noctl', 'Blues', 0.5, 1,
                    'R (No control, Peaks vs Pattern0 Random Seq)', 'R (No control, Peaks vs Dinuc Sh. Peaks)',
                    'AUROC No Control', guides=False)
fig.savefig(fig_path('r_noctl_p0rand_vs_shuf.pdf'))

In [ ]:
fig = family_panels('auroc_noctl', 'auroc_shufall_noctl', 'auprc_noctl', 'Blues', 0.5, 1,
                    'AUROC No Control', 'AUROC No Control Dinuc Shuffled Seqs', 'AUPRC No Control', guides=False)
fig.savefig(fig_path('auroc_noctl_vs_shufall.pdf'))